# 🏥 Medical LoRA Fine-tuning

Fine-tune Mistral-7B sur un dataset médical avec LoRA.

**Runtime**: Assure-toi que le GPU est activé (⚙️ Paramètres > GPU T4)

**Temps estimé**: 30-45 minutes

## Step 1: Installer les dépendances

In [1]:
!pip install -q torch transformers datasets peft bitsandbytes
print("✓ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
✓ Dependencies installed


## Step 2: Upload tes fichiers JSON

In [2]:
from google.colab import files
import os

print("Upload tes 3 fichiers JSON:")
print("  - medical_lora_train.json")
print("  - medical_lora_val.json")
print("  - medical_lora_test.json")
print()

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"✓ {filename} uploaded")

Upload tes 3 fichiers JSON:
  - medical_lora_train.json
  - medical_lora_val.json
  - medical_lora_test.json



Saving medical_lora_test.json to medical_lora_test.json
Saving medical_lora_train.json to medical_lora_train.json
Saving medical_lora_val.json to medical_lora_val.json
✓ medical_lora_test.json uploaded
✓ medical_lora_train.json uploaded
✓ medical_lora_val.json uploaded


## Step 3: Charger les données

In [3]:
import json
from datasets import Dataset

print("Chargement des données...")

with open('medical_lora_train.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)

with open('medical_lora_val.json', 'r', encoding='utf-8') as f:
    val_data = json.load(f)

print(f"✓ Train: {len(train_data)} exemples")
print(f"✓ Val: {len(val_data)} exemples")

# Format les données
def format_example(ex):
    return f"Q: {ex['instruction']}\nContext: {ex['input']}\nA: {ex['output']}"

texts_train = [format_example(ex) for ex in train_data]
texts_val = [format_example(ex) for ex in val_data]

train_dataset = Dataset.from_dict({"text": texts_train})
val_dataset = Dataset.from_dict({"text": texts_val})

print(f"\n✓ Datasets créés")

Chargement des données...
✓ Train: 191543 exemples
✓ Val: 23943 exemples

✓ Datasets créés


## Step 4: Tokenization

In [6]:
from transformers import AutoTokenizer

print("Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenization...")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing train"
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing val"
)

print("✓ Tokenization complète")

Chargement du tokenizer...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Tokenization...


Tokenizing train:   0%|          | 0/191543 [00:00<?, ? examples/s]

Tokenizing val:   0%|          | 0/23943 [00:00<?, ? examples/s]

✓ Tokenization complète


## Step 5: Charger le modèle avec Quantization

In [8]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print("Chargement du modèle TinyLlama avec 4-bit quantization...")
print("(Cela peut prendre 1-2 minutes)\n")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.pretraining_tp = 1

print("✓ Modèle TinyLlama chargé avec succès!")

Chargement du modèle TinyLlama avec 4-bit quantization...
(Cela peut prendre 1-2 minutes)



model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✓ Modèle TinyLlama chargé avec succès!


## Step 6: Configurer LoRA

In [9]:
from peft import get_peft_model, LoraConfig, TaskType

print("Configuration LoRA...\n")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

# Afficher les stats
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
percent = 100 * trainable_params / total_params

print(f"Trainable params: {trainable_params:,}")
print(f"Total params: {total_params:,}")
print(f"Percentage: {percent:.2f}%")
print("\n✓ LoRA configuré")

Configuration LoRA...

Trainable params: 2,252,800
Total params: 617,859,072
Percentage: 0.36%

✓ LoRA configuré


## Step 7: Fine-tuning

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

print("Configuration de l'entraînement...\n")

training_args = TrainingArguments(
    output_dir="./medical_lora_model",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=50,
    gradient_accumulation_steps=2,
    fp16=True,
    max_grad_norm=1.0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("🚀 Lancement du fine-tuning...")
print("(Cela prendra ~15-20 minutes avec TinyLlama)\n")

trainer.train()

Configuration de l'entraînement...

🚀 Lancement du fine-tuning...
(Cela prendra ~15-20 minutes avec TinyLlama)



Epoch,Training Loss,Validation Loss


## Step 8: Sauvegarder le modèle

In [ ]:
import os

print("Sauvegarde du modèle...\n")

output_dir = "./medical_lora_model"
os.makedirs(output_dir, exist_ok=True)

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("✓ Modèle sauvegardé\n")

# Afficher les fichiers
print("📦 Fichiers générés:")
total_size = 0
for f in os.listdir(output_dir):
    fpath = os.path.join(output_dir, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / (1024*1024)
        total_size += size
        print(f"  ✓ {f:<30} ({size:>7.2f} MB)")

print(f"\nTaille totale: {total_size:.2f} MB")
print("\n✅ Fine-tuning COMPLÉTÉ!")
print("✓ Ton .bin est dans: medical_lora_model/adapter_model.bin")

## Step 9: Télécharger le modèle

In [ ]:
from google.colab import files
import shutil
import os

print("Création du ZIP...")
shutil.make_archive('medical_lora_model', 'zip', '.', 'medical_lora_model')

zip_size = os.path.getsize('medical_lora_model.zip') / (1024*1024)
print(f"✓ ZIP créé ({zip_size:.2f} MB)")

print("\nTéléchargement...\n")
files.download('medical_lora_model.zip')

print("\n✅ Téléchargement terminé!")
print("\n📦 Fichiers dans le ZIP:")
print("  ✓ adapter_model.bin      ← TON FICHIER FINE-TUNÉ")
print("  ✓ adapter_config.json")
print("  ✓ config.json")
print("  ✓ tokenizer.model")

## Step 10 (Optionnel): Tester le modèle

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
import torch

print("Chargement du modèle fine-tuné pour test...\n")

model_test = AutoPeftModelForCausalLM.from_pretrained(
    "./medical_lora_model",
    device_map="auto",
    torch_dtype=torch.float32
)

tokenizer_test = AutoTokenizer.from_pretrained("./medical_lora_model")

# Exemple de test
test_prompts = [
    "Q: What causes sudden knee pain?",
    "Q: What are early signs of pregnancy?",
    "Q: What does trivial mitral regurgitation mean?"
]

print("Test d'inférence:\n")
for prompt in test_prompts:
    print(f"Prompt: {prompt}")

    inputs = tokenizer_test(prompt, return_tensors="pt").to(model_test.device)

    with torch.no_grad():
        outputs = model_test.generate(
            **inputs,
            max_length=100,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    response = tokenizer_test.decode(outputs[0], skip_special_tokens=True)
    print(f"Response: {response}")
    print("-" * 60)
    print()

print("✅ Test terminé!")

## ✅ Terminé!

Ton modèle fine-tuné est prêt!

**Ce que tu as:**
- ✓ `adapter_model.bin` - Les poids LoRA fine-tunés
- ✓ `adapter_config.json` - Configuration LoRA
- ✓ `config.json` - Configuration du modèle
- ✓ `tokenizer.model` - Tokenizer

**Utilisation:**
```python
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

model = AutoPeftModelForCausalLM.from_pretrained("./medical_lora_model")
tokenizer = AutoTokenizer.from_pretrained("./medical_lora_model")

inputs = tokenizer("Your prompt", return_tensors="pt")
outputs = model.generate(**inputs, max_length=200)
print(tokenizer.decode(outputs[0]))
```